# IQ on hg19 - 5mc vs nothing

In [1]:
library(prego)
library(iceqream)
options(tidyverse.quiet = TRUE)
library(tidyverse)
library(purrr)
library(tgutil)
library(misha)
library(misha.ext)
library(glue)
library(here)
library(tgstat)

gsetroot(here("data/hg19"))
options(gmax.data.size = 1e+9)
source(here("code/model-utils.R"))


Loading required package: misha


Attaching package: ‘misha.ext’


The following object is masked from ‘package:misha’:

    gintervals.normalize


here() starts at /net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper



In [2]:
cgd_all <- fread(here("output/cgd_hg19_iq.tsv")) 
rownames(cgd_all) <- cgd_all$id

In [3]:
pwm5 <- readr::read_rds(here("output/pwm5_norm_hg19_iq.rds"))
pwm5_pad <- readr::read_rds(here("output/pwm5_pad_norm_hg19_iq.rds"))
pwm3 <- readr::read_rds(here("output/pwm3_norm_hg19_iq.rds"))
pwm3_pad <- readr::read_rds(here("output/pwm3_pad_norm_hg19_iq.rds"))

In [4]:
pwm_in <- readr::read_rds(here("output/pwm_in_norm_hg19_iq.rds"))

In [5]:
add_feats <- readr::read_rds(here("output/add_feats_hg19_iq.rds"))

In [6]:
add_feats_in <- readr::read_rds(here("output/add_feats_in_hg19_iq.rds"))

We will start by training a predictor to distinguish between CGDD that have either K4me3 or K27me3, and then another predictor to distinguish between K27me3 and K4me3. 

In [7]:
test_chroms <- c("chr2", "chr8", "chr12", "chr18")
cgd_all <- cgd_all %>%
    mutate(f_500 = l > 500) %>%
    mutate(f_tss = tss_dist == 0) %>%
    mutate(k27_k4_l10 = log2(es_k27_max + es_k4_max + 10)) %>%
    mutate(resp10 = ifelse(k27_k4_l10 > 7, 1, ifelse(k27_k4_l10 < 4, 0, NA))) %>%
    mutate(orig_resp10 = resp10) %>%
    mutate(f_test = chrom %in% test_chroms) %>%
    mutate(f_train = !f_test) %>%
    mutate(f_ambig = is.na(resp10)) %>%
    mutate(type = case_when(
        f_500 & f_ambig ~ "ambig",
        f_500 & f_train ~ "train",
        f_500 & f_test ~ "test",
        TRUE ~ "other"
    ))

In [8]:
stopifnot(all(cgd_all$resp5mc[!is.na(cgd_all$resp5mc)] == cgd_all$resp10[!is.na(cgd_all$resp10)]))
stopifnot(all(is.na(cgd_all$resp5mc) == is.na(cgd_all$resp10)))


In [9]:
f <- cgd_all$type %in% c("train", "test") & cgd_all$id %in% rownames(pwm5) & !is.na(cgd_all$resp10) & cgd_all$id %in% rownames(pwm5_pad) & cgd_all$id %in% rownames(pwm3) & cgd_all$id %in% rownames(pwm3_pad) & cgd_all$id %in% rownames(add_feats)
sum(f)
cgd <- cgd_all[f, ]
fwrite(cgd, here("output/cgd_hg19.tsv"), sep = "\t")
pwm5 <- pwm5[rownames(cgd), ]
pwm5_pad <- pwm5_pad[rownames(cgd), ]
pwm3 <- pwm3[rownames(cgd), ]
pwm3_pad <- pwm3_pad[rownames(cgd), ]
add_feats <- add_feats[rownames(cgd), ]
add_feats_in <- add_feats_in[rownames(cgd), ]
pwm_in <- pwm_in[rownames(cgd), ]

[1] 18181

In [10]:
add_feats5 <- add_feats
colnames(add_feats5) <- gsub("_5$", "", colnames(add_feats5))
add_feats3 <- add_feats
colnames(add_feats3) <- gsub("_3$", "", colnames(add_feats3))
add_feats5_pad <- add_feats
colnames(add_feats5_pad) <- gsub("_5pad$", "", colnames(add_feats5_pad))
add_feats3_pad <- add_feats
colnames(add_feats3_pad) <- gsub("_3pad$", "", colnames(add_feats3_pad))

In [11]:
cgd5 <- cgd[, 1:4]
cgd5$start <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end - 600)
cgd5$end <- ifelse(cgd$tss_strand == 1, cgd$start + 600, cgd$end)

cgd5_pad <- cgd[, 1:4]
cgd5_pad$start <- ifelse(cgd$tss_strand == 1, cgd$start - 500, cgd$end)
cgd5_pad$end <- ifelse(cgd$tss_strand == 1, cgd$start, cgd$end + 500)

cgd3 <- cgd[, 1:4]
cgd3$start <- ifelse(cgd$tss_strand == 1, cgd$end - 600, cgd$start)
cgd3$end <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start + 600)

cgd3_pad <- cgd[, 1:4]
cgd3_pad$start <- ifelse(cgd$tss_strand == 1, cgd$end, cgd$start - 500)
cgd3_pad$end <- ifelse(cgd$tss_strand == 1, cgd$end + 500, cgd$start)

In [12]:
train_idxs <- which(cgd$type == "train")
test_idxs <- which(cgd$type == "test")

In [13]:
cgd5_oriented <- cgd5
cgd5_oriented$strand <- cgd$tss_strand
cgd5_oriented <- cgd5_oriented %>% select(chrom, start, end, strand, everything())

cgd5_pad_oriented <- cgd5_pad
cgd5_pad_oriented$strand <- cgd$tss_strand
cgd5_pad_oriented <- cgd5_pad_oriented %>% select(chrom, start, end, strand, everything())

cgd3_oriented <- cgd3
cgd3_oriented$strand <- cgd$tss_strand
cgd3_oriented <- cgd3_oriented %>% select(chrom, start, end, strand, everything())

cgd3_pad_oriented <- cgd3_pad
cgd3_pad_oriented$strand <- cgd$tss_strand
cgd3_pad_oriented <- cgd3_pad_oriented %>% select(chrom, start, end, strand, everything())

In [14]:
cgd_oriented <- cgd
cgd_oriented$strand <- cgd$tss_strand
cgd_oriented <- cgd_oriented %>% select(chrom, start, end, strand, everything())

## Run models

In [16]:
suffix <- "_prego_10_motifs_no_add_oriented_hg19_k27_k4_vs_nothing"
run_iq_regression(
    model_type = "5",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "5_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)
run_iq_regression(
    model_type = "3_pad",
    cgd = cgd,
    test_idxs = test_idxs,
    train_idxs = train_idxs,
    suffix = suffix
)

→ Computing sequence features

ℹ Seed: 60427

ℹ Training on 15128 intervals (83%) and testing on 3053 intervals (17%)

ℹ Saving the train and test indices to "/net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper/output/iq_5_prego_10_motifs_no_add_oriented_hg19_k27_k4_vs_nothing"

ℹ Learning prego motifs (de-novo motifs)

ℹ Extracting sequences...

ℹ Inferring 5 prego motifs...

ℹ Using 15 bins of size 40 bp

ℹ Running regression for 5 motifs



── Running first regression ──



ℹ Using 15 bins of size 40 bp

ℹ Using "ks" as the final metric

ℹ Number of response variables: 1

ℹ Performing sampled optimization

ℹ Sampling 1 of the dataset

ℹ Stratified sampling

ℹ sampled 6125 zeros and 9003 ones

ℹ Sampling 0.1 of the dataset

ℹ Stratified sampling

ℹ sampled 612 zeros and 900 ones



── Generate candidate kmers 



── Regress each candidate kmer on sampled data 

ℹ Running regression on 3 candidate kmers

• Bidirectional: TRUE

• Spat bin size: 40

• Num

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the pres

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the pres

...

ℹ Infering energies...

ℹ Running final regression, number of features: 108

✔ Finished running model. Number of non-zero coefficients: 107 (out of 108). R^2: 0.357930005335268

→ Matching SCENIC.fantom__motif138_CTTTWANNYCC

→ Matched with "HOMER.SEP3", PSSM correlation = 0.583231785960872

→ Matching JASPAR.Lhx4

→ Matched with "HOMER.IRF2", PSSM correlation = 0.594012106296296

→ Matching SCENIC.nitta__grn_TATCTG20NTCG_KA_WGATAASN_m1_c5

→ Matched with "HOMER.IRF2", PSSM correlation = 0.693268186753529

→ Matching SCENIC.predrem__nrMotif338

→ Matched with "HOMER.Smad2", PSSM correlation = 0.541762191576029

→ Matching JASPAR.CREB3L4.MA1474.1

→ Matched with "HOMER.E_box_2", PSSM correlation = 0.726573709781199

→ Matching SCENIC.predrem__nrMotif1820

→ Matched with "HOMER.SeqBias_A", PSSM correlation = 0.578306941962299

→ Matching SCENIC.cisbp__M11358

→ Matched with "HOMER.Mef2a", PSSM correlation = 0.561182925405623

→ Matching SCENIC.swissregulon__hs__SPIB

→ Matched with "HOME

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the pres

R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call
R_zmq_msg_send errno: 4 strerror: Interrupted system call


Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
Warning message in ks.test.default(plot_df$observed[plot_df$energy == "0-3"], plot_df$observed[plot_df$energy == :
“p-value will be approximate in the presence of ties”
ℹ Saving plot...

Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 46 rows containing missing values or values outside the scale range
(`geom_line()`).”
✔ Plot saved to /net/mraid20/ifs/wisdom/tanay_lab/tgdata/users/evghenic/Proj/polycomb/seq2epi_paper/output/iq_3_pad_prego_10_motifs_no_add_oriented_hg19_k27_k4_vs_nothing/iq_regression_filtered_mo